# 03 - Exploratory Data Analysis

## Online Retail Sales & Customer Analysis

This notebook explores the cleaned Online Retail transaction data to identify patterns in sales performance, products, customers, and geography.

The analysis builds on the data-understanding and cleaning stages. Rather than examining variables in isolation, the exploratory analysis focuses on questions that can provide meaningful insights into the business and guide subsequent customer-level analysis.

The main objectives are to:

* understand overall sales and transaction patterns;
* examine sales performance over time;
* identify important products and product-level patterns;
* explore geographic differences in sales;
* examine customer purchasing activity;
* identify patterns and questions that should be investigated in later analysis.

## 1. Load the Cleaned Dataset

The exploratory analysis is performed on the cleaned dataset produced in the data-cleaning stage.

The cleaning process removed exact duplicate records, standardized the `CustomerID` data type, and retained unusual transaction values where their meaning could not be determined reliably from the data alone.

Loading the cleaned dataset ensures that the exploratory analysis uses a consistent version of the data without repeating the cleaning process.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
# Load Data

df = pd.read_csv(
    "../data/processed/cleaned_online_retail.csv",
    parse_dates=["InvoiceDate"]
)

In [ ]:
df["InvoiceDate"].dtype

In [ ]:
df["InvoiceDate"].dt.month

In [ ]:
df["InvoiceDate"].dt.year

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df.isna().sum()

In [ ]:
df.head()

### 1.1 Restore CustomerID Data Type

When the cleaned dataset is saved as CSV and loaded again, Pandas may infer `CustomerID` as a floating-point column because the column contains missing values.

Since `CustomerID` is an identifier rather than a numerical measurement, it is converted back to Pandas' nullable `Int64` type for consistency with the cleaned dataset.

In [ ]:
df["CustomerID"] = df["CustomerID"].astype("Int64")

In [ ]:
df["CustomerID"].dtype

## 2. Overall Business Overview

Before examining individual products, customers, or countries, it is useful to establish a baseline view of the dataset.

This section summarizes the overall scale of the transactions and the period covered by the data. The purpose is to understand the size and scope of the business represented in the dataset before investigating more detailed patterns.

The analysis will examine:

* total net revenue;
* total transaction lines;
* total invoices;
* total units recorded;
* the time period covered by the dataset.

### 2.1 Data Period

The transaction data covers the period from **1 December 2010** to **9 December 2011**.

Understanding the time range is important for interpreting sales trends and identifying possible seasonal patterns. Since the dataset spans approximately one year, monthly and time-based comparisons can provide useful insight into how sales activity changes throughout the observed period.


In [ ]:
df["InvoiceDate"].min()

In [ ]:
df["InvoiceDate"].max()

### 2.2 Overall Business Activity

The cleaned dataset contains **536,641 transaction lines** across **25,900 unique invoices**. The total recorded quantity is **5,162,502 units**, and the resulting net revenue is approximately **£9.73 million**.

These figures provide a baseline for the scale of the business represented in the dataset. The transaction-line count is substantially higher than the invoice count because a single invoice can contain multiple transaction lines.

The revenue and quantity totals include negative transactions, which are retained because they may represent returns, cancellations, or other negative adjustments. Therefore, the revenue reported here represents **net recorded revenue** rather than gross sales before such transactions.

In [ ]:
df["Revenue"].sum()

In [ ]:
len(df)

In [ ]:
df["InvoiceNo"].nunique()

In [ ]:
df["Quantity"].sum()

### 2.3 Dataset Entities

The dataset contains **4,372 identified customers**, **4,070 unique products**, and transactions from **38 countries**.

The customer count represents only transactions with an identified `CustomerID`. Missing customer identifiers are not counted as customers because they cannot be reliably attributed to a specific customer.

These figures provide an overview of the main entities represented in the dataset and establish the baseline for the more detailed product, geographic, and customer analyses that follow.

In [ ]:
df["CustomerID"].nunique()

In [ ]:
df["StockCode"].nunique()

In [ ]:
df["Country"].nunique()

## 3. Sales Over Time

After establishing the overall scale of the dataset, the next step is to examine how sales activity changes over time.

Aggregating transactions by month allows us to identify overall trends, periods of increased or decreased activity, and potential seasonal patterns. This provides a clearer view of the business than examining individual transaction timestamps.

The initial focus will be on monthly revenue, followed by other measures of sales activity where useful.

In [ ]:
df["Month"] = df["InvoiceDate"].dt.to_period("M")

In [ ]:
monthly_revenue = df.groupby("Month")["Revenue"].sum()

In [ ]:
monthly_revenue

In [ ]:
len(monthly_revenue)

In [ ]:
plt.plot(monthly_revenue.index.astype(str), monthly_revenue)

plt.title("Monthly Revenue")
plt.xlabel("Month")
plt.ylabel("Revenue (£)")
plt.xticks(rotation=45)
plt.show()

In [ ]:
monthly_invoices = df.groupby("Month")["InvoiceNo"].nunique()

In [ ]:
monthly_invoices

In [ ]:
monthly_units = df.groupby("Month")["Quantity"].sum()

In [ ]:
monthly_units

In [ ]:
invoice_revenue = df.groupby(["Month", "InvoiceNo"])["Revenue"].sum()

In [ ]:
invoice_revenue

In [ ]:
monthly_avg_revenue_per_invoice = invoice_revenue.groupby("Month").mean()

In [ ]:
monthly_avg_revenue_per_invoice

### 3.1 Findings

Monthly revenue shows a relatively weaker sales period during the beginning of the year, followed by more stable activity from approximately May to August. Revenue then increases substantially from September through November, with November recording the highest monthly revenue in the dataset.

The increase in revenue during September–November is accompanied by increases in both the number of invoices and the number of units recorded. Average revenue per invoice also remains relatively high during this period, but does not increase as sharply as total revenue. This suggests that the increase in total revenue is primarily associated with higher sales activity and volume, rather than a large increase in revenue generated by each individual invoice.

December should be interpreted cautiously because the dataset only contains transactions through **9 December 2011**, making it a partial month rather than a complete comparison with the preceding months.


In [ ]:
df["Revenue"].describe()

In [ ]:
plt.boxplot(df["Revenue"])
plt.title("Distribution of Revenue per Transaction Line")
plt.xlabel("Revenue (£)")
plt.ylabel("Number of Transaction Lines")
plt.show()

### 3.2 Revenue Distribution

The revenue distribution is highly concentrated around relatively small transaction values, while a small number of observations have extremely large positive or negative values. These extreme observations stretch the scale of the histogram and make the distribution of typical transaction revenues difficult to observe.

The box plot confirms the presence of substantial outliers in transaction-level revenue. These observations are retained because they may represent legitimate high-value transactions, returns, or special transaction records identified during the data-cleaning stage.

In [ ]:
product_revenue = df.groupby("StockCode")["Revenue"].sum()

## to check

### revise this part

In [ ]:
product_revenue

In [ ]:
top_products = product_revenue.nlargest(20)

In [ ]:
df[df["StockCode"] == "DOT"][["StockCode", "Description", "Quantity", "UnitPrice", "Revenue"]].head(20)

In [ ]:
df[df["StockCode"].isin(top_products.index)][
    ["StockCode", "Description"]
].drop_duplicates().sort_values("StockCode")

In [ ]:
special_codes = ["DOT", "POST", "D", "M", "m", "AMAZONFEE", "B"]

In [ ]:
# New df excluding the special codes for non-products
product_df = df[~df["StockCode"].isin(special_codes)]

In [ ]:
product_df.columns

In [ ]:
product_df.shape

In [ ]:
product_revenue_clean = product_df.groupby("StockCode")["Revenue"].sum()

In [ ]:
product_revenue_clean.nlargest(10)

In [ ]:
product_df[product_df["StockCode"].isin(product_revenue_clean.nlargest(10).index)][
    ["StockCode", "Description"]
].drop_duplicates()

In [141]:
description_counts = (
    product_df
    .groupby(["StockCode", "Description"])
    .size()
    .sort_values(ascending=False)
)

In [142]:
description_counts

StockCode  Description                       
85123A     WHITE HANGING HEART T-LIGHT HOLDER    2290
22423      REGENCY CAKESTAND 3 TIER              2189
85099B     JUMBO BAG RED RETROSPOT               2156
47566      PARTY BUNTING                         1720
20725      LUNCH BAG RED RETROSPOT               1625
                                                 ... 
22700      smashed                                  1
22687      adjustment                               1
           had been put aside                       1
90210D     check                                    1
23530      WALL ART,ONLY ONE PERSON                 1
Length: 4785, dtype: int64

In [143]:
top_10_products = product_revenue_clean.nlargest(10)

top_10_descriptions = (
    description_counts
    .loc[
        description_counts.index.get_level_values("StockCode")
        .isin(top_10_products.index)
    ]
)

In [160]:
top_10_descriptions = (
    top_10_descriptions
    .groupby(level="StockCode")
    .head(1)
)

In [162]:
top_10_descriptions = (
    top_10_descriptions
    .groupby(level="StockCode", group_keys=False)
    .apply(lambda x: x.sort_values(ascending=False).head(1))
)

In [163]:
top_10_descriptions

StockCode  Description                       
85123A     WHITE HANGING HEART T-LIGHT HOLDER    2290
22423      REGENCY CAKESTAND 3 TIER              2189
85099B     JUMBO BAG RED RETROSPOT               2156
47566      PARTY BUNTING                         1720
84879      ASSORTED COLOUR BIRD ORNAMENT         1488
22086      PAPER CHAIN KIT 50'S CHRISTMAS        1194
23084      RABBIT NIGHT LIGHT                    1032
22197      POPCORN HOLDER                         861
79321      CHILLI LIGHTS                          676
22502      PICNIC BASKET WICKER SMALL             474
dtype: int64

In [165]:
top_10_product_table = (
    top_10_products.to_frame(name="Revenue")
    .merge(
        top_10_descriptions.reset_index(),
        left_index=True,
        right_on="StockCode"
    )
    [["StockCode", "Description", "Revenue"]]
)

In [166]:
top_10_product_table

,StockCode,Description,Revenue
1,22423,REGENCY CAKESTAND 3 TIER,164459.49
3,47566,PARTY BUNTING,98243.88
0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,97838.45
2,85099B,JUMBO BAG RED RETROSPOT,92175.79
6,23084,RABBIT NIGHT LIGHT,66661.63
5,22086,PAPER CHAIN KIT 50'S CHRISTMAS,63715.24
4,84879,ASSORTED COLOUR BIRD ORNAMENT,58792.42
8,79321,CHILLI LIGHTS,53746.66
9,22502,PICNIC BASKET WICKER SMALL,51023.52
7,22197,POPCORN HOLDER,50967.92


In [167]:
top_10_product_table = top_10_product_table.sort_values(
    "Revenue",
    ascending=False
).reset_index(drop=True)

In [168]:
top_10_product_table

,StockCode,Description,Revenue
0,22423,REGENCY CAKESTAND 3 TIER,164459.49
1,47566,PARTY BUNTING,98243.88
2,85123A,WHITE HANGING HEART T-LIGHT HOLDER,97838.45
3,85099B,JUMBO BAG RED RETROSPOT,92175.79
4,23084,RABBIT NIGHT LIGHT,66661.63
5,22086,PAPER CHAIN KIT 50'S CHRISTMAS,63715.24
6,84879,ASSORTED COLOUR BIRD ORNAMENT,58792.42
7,79321,CHILLI LIGHTS,53746.66
8,22502,PICNIC BASKET WICKER SMALL,51023.52
9,22197,POPCORN HOLDER,50967.92


### revise this part